# Transfer learning para patrones geométricos

Clasificaremos patrones de vasijas griegas con pocos datos usando ResNet18 preentrenada en ImageNet.

**Conceptos:** transferencia de representaciones, aumento de datos, feature extraction, fine-tuning, validación y scheduler.

**Resultado esperado:** entrenar solo la capa final es más rápido; ajustar toda la red ofrece más flexibilidad, pero también mayor costo y riesgo de sobreajuste.


## 1. Contexto y preparación de datos

El conjunto contiene 348 patrones agrupados en seis clases. En vez de copiar imágenes a carpetas train/val/test, construiremos listas de rutas con una división estratificada. Esto evita archivos duplicados y mantiene la partición reproducible.


In [ ]:
from pathlib import Path
from shutil import copyfileobj
from urllib.request import Request, urlopen
from zipfile import ZipFile

archive_path = Path("data_patterns2.zip")
labels_path = Path("data/class_labels.csv")

if not labels_path.exists():
    if not archive_path.exists():
        download_request = Request(
            "http://www.ivan-sipiran.com/downloads/data_patterns2.zip",
            headers={"User-Agent": "Mozilla/5.0"},
        )
        with urlopen(download_request, timeout=60) as response, archive_path.open("wb") as output_file:
            copyfileobj(response, output_file)
    with ZipFile(archive_path) as archive:
        archive.extractall(".")

if not labels_path.exists():
    raise FileNotFoundError(
        "No se encontró data/class_labels.csv después de extraer los datos."
    )

print("Datos disponibles en:", labels_path.parent.resolve())


In [ ]:
import copy
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

SEED = 30
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", device)


## 2. Partición estratificada

Una partición estratificada conserva aproximadamente la proporción de cada clase. Usamos 70 % train, 10 % validación y 20 % test.

Validación sirve para seleccionar el modelo; test se reserva para la comparación final.


In [ ]:
label_table = pd.read_csv(labels_path, header=None)
sample_ids = label_table[0].astype(str).to_numpy()
sample_labels = label_table[1].astype(str).to_numpy()

class_names = sorted(np.unique(sample_labels))
class_to_index = {
    class_name: index
    for index, class_name in enumerate(class_names)
}

image_paths = np.array(
    [
        Path("data/patrones") / sample_id / f"{sample_id}_pattern.png"
        for sample_id in sample_ids
    ],
    dtype=object,
)
missing_paths = [path for path in image_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        f"Faltan {len(missing_paths)} imágenes. Primera: {missing_paths[0]}"
    )

all_indices = np.arange(len(image_paths))
train_indices, temporary_indices = train_test_split(
    all_indices,
    test_size=0.30,
    random_state=SEED,
    stratify=sample_labels,
)
validation_indices, test_indices = train_test_split(
    temporary_indices,
    test_size=2 / 3,
    random_state=SEED,
    stratify=sample_labels[temporary_indices],
)

print("Clases:", class_names)
print("Train:", len(train_indices))
print("Validación:", len(validation_indices))
print("Test:", len(test_indices))


## 3. Dataset, transformaciones y mini-batches

Train usa recorte, rotación leve y espejo horizontal para exponer variaciones plausibles. Validación y test usan transformaciones deterministas. Todas las imágenes se normalizan como ImageNet porque utilizaremos esos pesos.

**Resultado esperado:** tensores de forma (batch, 3, 224, 224) y etiquetas enteras.


In [ ]:
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose(
    [
        transforms.RandomResizedCrop(224, scale=(0.80, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize(imagenet_mean, imagenet_std),
    ]
)
evaluation_transform = transforms.Compose(
    [
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(imagenet_mean, imagenet_std),
    ]
)

class PatternDataset(Dataset):
    def __init__(self, indices, transform):
        self.indices = list(indices)
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        sample_index = self.indices[item]
        image = Image.open(image_paths[sample_index]).convert("RGB")
        image_tensor = self.transform(image)
        label_index = class_to_index[sample_labels[sample_index]]
        return image_tensor, label_index


train_dataset = PatternDataset(train_indices, train_transform)
validation_dataset = PatternDataset(
    validation_indices,
    evaluation_transform,
)
test_dataset = PatternDataset(test_indices, evaluation_transform)

BATCH_SIZE = 16
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    generator=torch.Generator().manual_seed(SEED),
)
validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

batch_images, batch_labels = next(iter(train_loader))
print("Imágenes:", batch_images.shape)
print("Etiquetas:", batch_labels.shape)


In [ ]:
def undo_imagenet_normalization(image_tensor):
    # Revierte la normalización solo para mostrar una imagen.
    mean = torch.tensor(imagenet_mean).view(3, 1, 1)
    std = torch.tensor(imagenet_std).view(3, 1, 1)
    return torch.clamp(image_tensor.cpu() * std + mean, 0, 1)


figure, axes = plt.subplots(2, 4, figsize=(12, 6))
for image, label, axis in zip(
    batch_images[:8],
    batch_labels[:8],
    axes.ravel(),
):
    axis.imshow(
        undo_imagenet_normalization(image).permute(1, 2, 0)
    )
    axis.set_title(class_names[label.item()])
    axis.axis("off")
plt.tight_layout()
plt.show()


## 4. Ciclo común de entrenamiento

run_epoch separa entrenamiento de evaluación mediante la presencia del optimizador. fit_model conserva el estado con mejor accuracy de validación y, si recibe un scheduler, lo actualiza al final de cada época.

**Resultado esperado:** los logs muestran pérdida, accuracy y learning rate por época sin imprimir cada mini-batch.


In [ ]:
def run_epoch(model, data_loader, criterion, device, optimizer=None):
    is_training = optimizer is not None
    model.train(mode=is_training)

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for images, labels in data_loader:
        images = images.to(device)
        labels = labels.to(device)

        if is_training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_training):
            logits = model(images)
            loss = criterion(logits, labels)
            if is_training:
                loss.backward()
                optimizer.step()

        batch_size = images.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_samples += batch_size

    return {
        "loss": total_loss / total_samples,
        "accuracy": total_correct / total_samples,
    }


def fit_model(
    model,
    train_loader,
    validation_loader,
    criterion,
    optimizer,
    epochs,
    scheduler=None,
):
    best_state = copy.deepcopy(model.state_dict())
    best_validation_accuracy = 0.0
    history = []

    for epoch in range(1, epochs + 1):
        start_time = time.time()
        train_metrics = run_epoch(
            model,
            train_loader,
            criterion,
            device,
            optimizer,
        )
        validation_metrics = run_epoch(
            model,
            validation_loader,
            criterion,
            device,
        )

        if validation_metrics["accuracy"] > best_validation_accuracy:
            best_validation_accuracy = validation_metrics["accuracy"]
            best_state = copy.deepcopy(model.state_dict())

        current_learning_rate = optimizer.param_groups[0]["lr"]
        history.append(
            {
                "train": train_metrics,
                "validation": validation_metrics,
                "learning_rate": current_learning_rate,
            }
        )
        print(
            f"Época {epoch:02d}/{epochs} | "
            f"train acc {train_metrics['accuracy']:.3f} | "
            f"val acc {validation_metrics['accuracy']:.3f} | "
            f"lr {current_learning_rate:.1e} | "
            f"{time.time() - start_time:.1f} s"
        )

        if scheduler is not None:
            scheduler.step()

    model.load_state_dict(best_state)
    print(
        "Mejor accuracy de validación:",
        f"{100 * best_validation_accuracy:.2f} %",
    )
    return model, history


## 5. Estrategia A: feature extraction

Congelamos el backbone y reemplazamos la capa final. Solo esa capa aprende la nueva tarea; por eso el optimizador recibe únicamente parámetros con requires_grad=True.


**Feature extraction:** imagen → ResNet18 congelada → vector de 512 features →
nueva capa de seis clases entrenable.

El gradiente se calcula para la capa final; el backbone actúa como extractor fijo.


In [ ]:
def create_resnet18_classifier(num_classes, freeze_backbone):
    weights = models.ResNet18_Weights.DEFAULT
    model = models.resnet18(weights=weights)

    if freeze_backbone:
        for parameter in model.parameters():
            parameter.requires_grad = False

    input_features = model.fc.in_features
    model.fc = nn.Linear(input_features, num_classes)
    return model


feature_model = create_resnet18_classifier(
    num_classes=len(class_names),
    freeze_backbone=True,
).to(device)

feature_trainable_parameters = sum(
    parameter.numel()
    for parameter in feature_model.parameters()
    if parameter.requires_grad
)
print(
    f"Parámetros entrenables: {feature_trainable_parameters:,}"
)

criterion = nn.CrossEntropyLoss()
feature_optimizer = torch.optim.Adam(
    (
        parameter
        for parameter in feature_model.parameters()
        if parameter.requires_grad
    ),
    lr=1e-3,
)

EPOCHS = 5
feature_model, feature_history = fit_model(
    feature_model,
    train_loader,
    validation_loader,
    criterion,
    feature_optimizer,
    EPOCHS,
)


## 6. Estrategia B: fine-tuning con scheduler

Ahora todos los parámetros pueden cambiar. Usamos una tasa menor para no destruir rápidamente las representaciones preentrenadas. StepLR reduce el learning rate después de tres épocas.

**Resultado esperado:** habrá muchos más parámetros entrenables y cada época tardará más.


In [ ]:
finetune_model = create_resnet18_classifier(
    num_classes=len(class_names),
    freeze_backbone=False,
).to(device)

finetune_trainable_parameters = sum(
    parameter.numel()
    for parameter in finetune_model.parameters()
    if parameter.requires_grad
)
print(
    f"Parámetros entrenables: {finetune_trainable_parameters:,}"
)

finetune_optimizer = torch.optim.AdamW(
    finetune_model.parameters(),
    lr=1e-4,
)
learning_rate_scheduler = torch.optim.lr_scheduler.StepLR(
    finetune_optimizer,
    step_size=3,
    gamma=0.1,
)

finetune_model, finetune_history = fit_model(
    finetune_model,
    train_loader,
    validation_loader,
    criterion,
    finetune_optimizer,
    EPOCHS,
    scheduler=learning_rate_scheduler,
)


## 7. Evaluación en test

Recién ahora usamos test. Esta comparación es descriptiva: con un dataset pequeño, una sola partición puede tener alta varianza y no basta para afirmar superioridad general.


In [ ]:
feature_test_metrics = run_epoch(
    feature_model,
    test_loader,
    criterion,
    device,
)
finetune_test_metrics = run_epoch(
    finetune_model,
    test_loader,
    criterion,
    device,
)

print(
    "Feature extraction | "
    f"loss {feature_test_metrics['loss']:.4f} | "
    f"accuracy {100 * feature_test_metrics['accuracy']:.2f} %"
)
print(
    "Fine-tuning        | "
    f"loss {finetune_test_metrics['loss']:.4f} | "
    f"accuracy {100 * finetune_test_metrics['accuracy']:.2f} %"
)

output_dir = Path("outputs/transfer_patterns")
output_dir.mkdir(parents=True, exist_ok=True)
torch.save(
    feature_model.state_dict(),
    output_dir / "resnet18_feature_extractor.pt",
)
torch.save(
    finetune_model.state_dict(),
    output_dir / "resnet18_finetuned.pt",
)
print("Checkpoints guardados en:", output_dir)


## 8. Ejercicios

**Ejercicio 1 — Ablación de aumentos.** Elimine RandomRotation y luego RandomHorizontalFlip, manteniendo semilla y partición. Compare validación.

**Resultado esperado:** un aumento ayuda solo si representa una invariancia razonable para la tarea.

**Ejercicio 2 — Descongelamiento parcial.** Congele todo y descongele únicamente layer4 y fc de ResNet18.

**Resultado esperado:** el número de parámetros y costo quedan entre feature extraction y fine-tuning completo.

**Ejercicio 3 — Variabilidad.** Repita con tres semillas y reporte media y desviación estándar de accuracy en test.

**Resultado esperado:** en conjuntos pequeños la variabilidad entre particiones puede ser relevante; no se debe escoger la mejor semilla.
